In [ ]:
from pathlib import Path
import runpy

training_root = next(
    (parent / "back-end/ia/training" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "back-end/ia/training").is_dir()),
    None,
)
if training_root is None:
    raise RuntimeError("Execute este notebook dentro do repositório WaterPath_TCC.")
dataset_path = training_root / "datasets/computer_vision/object_detection"
base_path = dataset_path
script_path = training_root / "scripts/computer_vision/split_dataset.py"
split = runpy.run_path(str(script_path))["split_dataset"]
print(split(dataset_path))


In [ ]:
!pip install ultralytics

In [ ]:
import ultralytics

print(ultralytics.__version__)

In [ ]:
from ultralytics import YOLO

model = YOLO(str(training_root / "base_models/computer_vision/yolov8n.pt"))
model.train(
    data=str(base_path / "data.yaml"),
    epochs=20,
    imgsz=800,
    project=str(training_root / "experiment_outputs/computer_vision/detect"),
)
training_run = Path(model.trainer.save_dir)
best_weights = Path(model.trainer.best)


In [ ]:
# Avalia os melhores pesos no conjunto de teste reservado.
model = YOLO(str(best_weights))
model.val(data=str(base_path / "data.yaml"), split="test")


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(training_run / "results.png")))
display(Image(filename=str(training_run / "BoxP_curve.png")))


In [ ]:
# Exemplo de inferência em uma imagem do conjunto de teste.
test_image = (base_path / "test.txt").read_text(encoding="utf-8").splitlines()[0]
image_path = base_path / test_image.removeprefix("./")
result = model.predict(str(image_path), show=True, save=False, conf=0.25)
